In [1]:
import pandas as pd
import numpy as np
import re

Importujeme slovenský SWN Dataframe, ktorý sme vytvorili v predchádzajúcom kroku.

In [2]:
swn_syn = pd.read_csv("SlovakSWN_DF.csv", index_col=0)

In [3]:
swn_syn.head()

,pos,sid,pos_score,neg_score,obj_score,synset_terms,gloss
synset_id,,,,,,,
a:00001740,a,1740,0.125,0.00,0.875,schopné#1,"""(usually followed by `to') having the necessa..."
a:00002098,a,2098,0.000,0.75,0.250,neschopný#1,"""(usually followed by `to') not having the nec..."
a:00002312,a,2312,0.000,0.00,1.000,dorzálne#2 abaxiálne#1,"""facing away from the axis of an organ or orga..."
a:00002527,a,2527,0.000,0.00,1.000,ventrálna#2 adaxiálne#1,"""nearest to or facing toward the axis of an or..."
a:00002730,a,2730,0.000,0.00,1.000,akroskopické#1,facing or on the side toward the apex


Výpočet polarity, agregácia synsetov.

In [4]:
#calculate polarity for each synset
swn_syn["polarity"] = swn_syn["pos_score"] - swn_syn["neg_score"]

#separate synset terms
tmp = swn_syn[["synset_terms", "polarity"]].copy()
#one synset term per row
tmp["term"] = tmp["synset_terms"].str.split()
tmp = tmp.explode("term")

#remove sense numbers
tmp["word"] = tmp["term"].str.split("#").str[0].str.lower().str.strip()
tmp = tmp.dropna(subset=["word"])
tmp = tmp[tmp["word"] != ""]

#aggregate synsets
lex_agg = tmp.groupby("word", as_index=False)["polarity"].mean()
lex_dict = dict(zip(lex_agg["word"], lex_agg["polarity"]))

print("Synsets:", len(swn_syn))
print("Unique words in lexicon:", len(lex_dict))

Synsets: 117659
Unique words in lexicon: 134847


SlovakSA

Importujeme SlovakSA spolu s lex_feats, ktoré sme vypočítali pri modelovaní RoBERTa.

In [5]:
train_df = pd.read_csv("slovakSA_lemma_train_lex.csv")
test_df = pd.read_csv("slovakSA_lemma_test_lex.csv")
val_df = pd.read_csv("slovakSA_lemma_val_lex.csv")

In [6]:
train_df.head()

,label,text,lex_feats
0,1,"Bola ústretová,milá a komunikatívna .Šikovná s...","[0.6289682388305664, 0.08985260874032974, 7.0,..."
1,1,"dobrá obsluha, pekná baba.","[1.0018938779830933, 0.2504734694957733, 4.0, ..."
2,1,"Prijemne vystupovanie, ochota pomôcť","[0.1845238208770752, 0.0615079402923584, 3.0, ..."
3,1,Za ochotu,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,1,Ďakujem za rýchle vybavenie a ústretovosť,"[0.011408727616071701, 0.0028521819040179253, ..."


Konverzia lex_feats na číselné hodnoty

In [7]:
import ast

def parse_lex_feats(df):
    df["lex_feats"] = df["lex_feats"].apply(ast.literal_eval)
    return df

train_df = parse_lex_feats(train_df)
val_df   = parse_lex_feats(val_df)
test_df  = parse_lex_feats(test_df)

print(type(train_df["lex_feats"].iloc[0]))
print(train_df["lex_feats"].iloc[0])

<class 'list'>
[0.6289682388305664, 0.08985260874032974, 7.0, 0.875, 0.9345238208770752, -0.3055555522441864, 0.5416666865348816, -0.25, 1.0, 0.0]


Support Vector Machine

In [8]:
X_train = np.array(train_df["lex_feats"].tolist())
y_train = train_df["label"].values

X_val   = np.array(val_df["lex_feats"].tolist())
y_val   = val_df["label"].values

X_test  = np.array(test_df["lex_feats"].tolist())
y_test  = test_df["label"].values

print(X_train.shape)

(3560, 10)


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

Použijeme StandardScaler na normalizáciu vstupu

In [10]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0, random_state=42)
svm.fit(X_train_s, y_train)

LinearSVC(random_state=42)

In [11]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

Cs = [0.01, 0.1, 1.0, 3.0, 10.0]
best = {"C": None, "f1": -1, "acc": None}

for C in Cs:
    clf = LinearSVC(C=C, random_state=42)
    clf.fit(X_train_s, y_train)
    pred = clf.predict(X_val_s)
    f1 = f1_score(y_val, pred)
    acc = accuracy_score(y_val, pred)
    if f1 > best["f1"]:
        best = {"C": C, "f1": f1, "acc": acc}

best

{'C': 0.01, 'f1': 0.9443254817987152, 'acc': 0.9003831417624522}

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

svm = LinearSVC(C=best["C"], class_weight="balanced", random_state=42)
svm.fit(X_train_s, y_train)

test_pred = svm.predict(X_test_s)

print("TEST Accuracy:", accuracy_score(y_test, test_pred))
print("TEST F1:", f1_score(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, digits=4))

TEST Accuracy: 0.8301343570057581
TEST F1: 0.8969132207338381
[[ 95  33]
 [144 770]]
              precision    recall  f1-score   support

           0     0.3975    0.7422    0.5177       128
           1     0.9589    0.8425    0.8969       914

    accuracy                         0.8301      1042
   macro avg     0.6782    0.7923    0.7073      1042
weighted avg     0.8899    0.8301    0.8503      1042



Extrahovanie váh pre interpretáciu

In [13]:
feature_names = [
    "pol_sum",            # 0 vals.sum()
    "pol_mean",           # 1 vals.mean()
    "hit_count",          # 2 len(vals)
    "hit_rate",           # 3 len(vals)/n
    "pos_sum",            # 4 pos.sum()
    "neg_sum",            # 5 neg.sum()
    "max_pos",            # 6 pos.max()
    "min_neg",            # 7 neg.min()  (najviac negatívne = minimum)
    "strong_pos_count",   # 8 (pos >= thr).sum()
    "strong_neg_count"    # 9 (neg <= -thr).sum()
]

In [14]:
weights = svm.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "weight": weights
}).sort_values("weight", ascending=False)

coef_df

,feature,weight
4,pos_sum,0.209174
7,min_neg,0.190486
1,pol_mean,0.152728
0,pol_sum,0.149863
3,hit_rate,0.086120
9,strong_neg_count,0.043623
6,max_pos,-0.019515
8,strong_pos_count,-0.024489
5,neg_sum,-0.087403
2,hit_count,-0.559644


Hodnota hit_count ukazuje, že čím viac sentimentovo hodnotených slov sa nachádza v texte, tým viac model inklinuje k negatívnej klasifikácii. Môže to znamenať napríklad že negatívne recenzie sú väčšinou dlhšie ako pozitívne.

In [15]:
pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 14.2 MB/s eta 0:00:00


Definícia triedy modelu, pre automatické ohodnotenie textu.

In [16]:
import stanza
stanza.download("sk", verbose=False)

#initialize stanza tokenizer
nlp = stanza.Pipeline(
    lang="sk",
    processors="tokenize,pos,lemma",
    tokenize_no_ssplit=True,
    use_gpu=True,
    verbose=False
)

In [17]:
def lemmatize_text(text):
    doc = nlp(text)
    lemmas = []
    for sent in doc.sentences:
        for w in sent.words:
            if w.lemma:
                lemmas.append(w.lemma.lower().replace(" ", "_"))
    return " ".join(lemmas)

In [18]:
#regex to take only alphabet characters
WORD_RE = re.compile(r"[A-Za-zÁÄČĎÉÍĹĽŇÓÔŔŠŤÚÝŽáäčďéíĺľňóôŕšťúýž]+", re.UNICODE)

#if text is string, find all matching regex
def lex_tokens(text: str):
    return WORD_RE.findall(text.lower()) if isinstance(text, str) else []

#extract 10 features from text
def lex_features(text: str, strong_thr: float = 0.5):
    toks = lex_tokens(text)
    n = len(toks)

    #safety check
    if n == 0:
        return np.zeros(10, dtype=np.float32)

    #take values from lexicon for each hit
    vals = [lex_dict[t] for t in toks if t in lex_dict]

    #if no match was found, fill with zeroes
    if not vals:
        return np.zeros(10, dtype=np.float32)

    #assign positive and negative values
    vals = np.array(vals, dtype=np.float32)
    pos = vals[vals > 0]
    neg = vals[vals < 0]

    #calculate features
    return np.array([
        vals.sum(),                     #polarity sum
        vals.mean(),                    #polarity mean
        len(vals),                      #hits
        len(vals) / n,                  #coverage
        pos.sum() if pos.size else 0,   #sum positive
        neg.sum() if neg.size else 0,   #sum negative
        pos.max() if pos.size else 0,   #max positive
        neg.min() if neg.size else 0,   #max negative
        (pos >= strong_thr).sum(),      #max strong positive
        (neg <= -strong_thr).sum()      #max strong negative
    ], dtype=np.float32)

In [ ]:
class SlovakSentimentModel:

    def __init__(self, svm_model, scaler=None):
        self.svm = svm_model
        self.scaler = scaler

    def predict(self, text):
      #lemmatize text and extract features
      lem_text = lemmatize_text(text)
      feats = lex_features(lem_text)
      X = feats.reshape(1, -1)
      X = self.scaler.transform(X)
      
      #predict sentiment and assign pos/neg value
      pred = int(self.svm.predict(X)[0])
      margin = float(self.svm.decision_function(X)[0])

      label = "positive" if pred == 1 else "negative"

      return {
          "label": label,
          "class": pred,
          "confidence": round(margin, 3)
      }

In [20]:
model = SlovakSentimentModel(svm, scaler)

In [21]:
print(model.predict("Milujem tento film."))
print(model.predict("Nenávidím tento film."))

{'label': 'positive', 'class': 1, 'confidence': 1.155}
{'label': 'negative', 'class': 0, 'confidence': -0.966}
